In [31]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import plotly.graph_objects as go
from rod.rod_generator import RodGenerator

In [32]:
def plot_cls(poses):
    """
    Creates an interactive visualization of multiple centerlines.

    Parameters:
    - poses: [(n_sites, 3)] Centerline positions
    """
    # Create figure
    fig = go.Figure()

    # Plot each helix
    for idx, pos in enumerate(poses):
        # Plot centerline
        fig.add_trace(go.Scatter3d(
            x=pos[:, 0], y=pos[:, 1], z=pos[:, 2],
            mode='lines',
            line=dict(width=3),
            name=f'Centerline {idx + 1}'
        ))


    # Update layout
    fig.update_layout(
        scene=dict(
            xaxis=dict(title='X'),
            yaxis=dict(title='Y'),
            zaxis=dict(title='Z'),
            aspectmode='data'
        ),
        title=dict(text='CL Visualization', y=0.95, x=0.5, xanchor='center', yanchor='top'),
    )

    fig.show()

In [33]:
'''
Input: set of centerlines on a scalp

Algorithm:
- normalize relative positions of root point from each CL on the scalp on some surface
- compute negative density gradient for each CL root point on the surface
- calculate force on each CL point based on density gradient
   - force magnitude falls off with distance from root point, but has same directionality
   - root points DO NOT MOVE, but other points do
- update CL points based on force
- repeat until equilibrium configuration is reached (loss will be some "energy" related to the centerline force)

Output: set of CLs with updated positions
'''

# generating a set of CLs
poses = [] # n x 3
thetas = []
for i in range(200):
    pos, theta = RodGenerator.straight_rod(20)
    pos[:, 0] += 5 * 5 * np.random.sample() # spacing out x coords
    pos[:, 1] += 5 * 5 * np.random.sample() # spacing out y coords
    poses.append(pos)
    thetas.append(theta)

roots = np.array([pos[0] for pos in poses])
poses = np.array(poses)

plot_cls(poses)

In [34]:
# around each centerline, create a bounding inverted cone

# Create meshgrid
L = 12
n = 100
x = np.linspace(-L, L, n)
y = np.linspace(-L, L, n)
z = np.linspace(-L, L, n)

X, Y, Z = np.meshgrid(x, y, z, indexing='ij')  # 'ij' indexing
values = np.zeros_like(X)
mesh_with_values = np.stack([X, Y, Z, values], axis=-1)  # (n, n, n, 4)

print(mesh_with_values.shape)
print(poses.shape)  # poses shape = (num_centerlines, points_per_cl, 3)

# Helper functions
def compute_tangents(poses_cl):
    """ Vectorized tangent computation along a centerline """
    tangents = np.zeros_like(poses_cl)
    tangents[0] = poses_cl[1] - poses_cl[0]
    tangents[-1] = poses_cl[-1] - poses_cl[-2]
    tangents[1:-1] = poses_cl[2:] - poses_cl[:-2]
    norms = np.linalg.norm(tangents, axis=-1, keepdims=True)
    return tangents / norms

def define_plane_vectors(tangent_vecs):
    """ Vectorized normal plane construction """
    # Choose a different arbitrary vector depending on tangent
    condition = np.abs(tangent_vecs[:, 0]) < np.abs(tangent_vecs[:, 1])
    u = np.zeros_like(tangent_vecs)
    u[condition] = np.array([1, 0, 0])
    u[~condition] = np.array([0, 1, 0])

    v = np.cross(tangent_vecs, u)
    v /= np.linalg.norm(v, axis=-1, keepdims=True)

    u = np.cross(tangent_vecs, v)
    u /= np.linalg.norm(u, axis=-1, keepdims=True)

    return u, v  # both (N, 3)

def define_local_circle_batch(u, v, centers, radii, points=100):
    """ Create circles at all centers in batch """
    theta = np.linspace(0, 2*np.pi, points)  # (points,)
    cos_theta = np.cos(theta)  # (points,)
    sin_theta = np.sin(theta)  # (points,)

    # Broadcasting shapes:
    # centers: (N, 3)
    # u, v: (N, 3)
    # cos_theta: (points,), sin_theta: (points,)
    circles = (centers[:, None, :] + 
               radii[:, None, None] * (cos_theta[None, :, None] * u[:, None, :] + sin_theta[None, :, None] * v[:, None, :]))
    # circles shape: (N, points, 3)
    return circles

# MAIN loop over centerlines
num_centerlines, points_per_cl, _ = poses.shape
circle_pts = np.zeros((num_centerlines, points_per_cl, 100, 3))  # (CLS, points, points_on_circle, xyz)

for cl in range(num_centerlines):
    cl_points = poses[cl]  # (points_per_cl, 3)
    
    tangents = compute_tangents(cl_points)  # (points_per_cl, 3)
    u, v = define_plane_vectors(tangents)  # (points_per_cl, 3) each
    
    # Radius scales with distance from root point
    root_point = cl_points[-1]
    dists = np.linalg.norm(root_point - cl_points, axis=-1)  # (points_per_cl,)
    radii = 0.5 * dists  # (points_per_cl,)

    circles = define_local_circle_batch(u, v, cl_points, radii, points=100)  # (points_per_cl, 100, 3)
    circle_pts[cl] = circles  # assign

print(circle_pts.shape)
print(poses[0].shape) # (22, 3)
print(circle_pts[0].shape) # (22, 100, 3)  
circle_pts0_reshaped = circle_pts[0].reshape(-1, 3)
circle_pts1_reshaped = circle_pts[1].reshape(-1, 3)
circle_pts2_reshaped = circle_pts[2].reshape(-1, 3)
combined_pts = np.vstack([poses[0], poses[1], poses[2], circle_pts0_reshaped, circle_pts1_reshaped, circle_pts2_reshaped])
plot_cls([combined_pts])

(100, 100, 100, 4)
(200, 22, 3)
(200, 22, 100, 3)
(22, 3)
(22, 100, 3)


In [35]:
# Start here - construct radially outwards field from CL within each bounding cone

# loop over centerlines & corresponding circles
for cl in range(poses.shape[0]):
    cl_points = circle_pts[cl] #(22, 100, 3)
    

In [40]:
# original 1-D axes you built once:
#   x.shape = y.shape = z.shape = (n,)    (monotonically increasing)
dx = x[1] - x[0]            # voxel spacing (same for x, y, z in your setup)

# bounding-box edges from the earlier step
xmin, xmax = mins[0] - pad, maxs[0] + pad
ymin, ymax = mins[1] - pad, maxs[1] + pad
zmin, zmax = mins[2] - pad, maxs[2] + pad

# searchsorted returns the first / last index whose coordinate meets the bound
i_min = np.searchsorted(x, xmin, side="left")
i_max = np.searchsorted(x, xmax, side="right")   # one past the last valid i
j_min = np.searchsorted(y, ymin, side="left")
j_max = np.searchsorted(y, ymax, side="right")
k_min = np.searchsorted(z, zmin, side="left")
k_max = np.searchsorted(z, zmax, side="right")

# sub-volume centred on the cones; views, not copies
X_sub = X[i_min:i_max, j_min:j_max, k_min:k_max]
Y_sub = Y[i_min:i_max, j_min:j_max, k_min:k_max]
Z_sub = Z[i_min:i_max, j_min:j_max, k_min:k_max]

coords = np.stack((X_sub.ravel(),
                   Y_sub.ravel(),
                   Z_sub.ravel()), axis=1)      # (N_keep, 3)

print("kept voxels:", coords.shape[0], "of", X.size)

kept voxels: 4815360 of 4851840


In [41]:
import numpy as np
from scipy.spatial import cKDTree

# --- 0.  flatten the grid ---------------------------------------------------
# coords = np.stack((X.ravel(), Y.ravel(), Z.ravel()), axis=1)   # (N,3)
field  = np.zeros(coords.shape[0], dtype=np.float32)

# --- 1.  pre-compute KD-trees, one per centre-line --------------------------
trees         = []
radii_per_cl  = []
for cl in range(num_centerlines):
    pts   = poses[cl]                        # (P,3)
    trees.append(cKDTree(pts))
    # radii has shape (P,) already from your earlier loop
    radii_per_cl.append(0.5 * np.linalg.norm(pts[-1] - pts, axis=1))

# --- 2.  kernel function ----------------------------------------------------
def kernel(r):
    inside = r < 1.0
    out    = np.zeros_like(r, dtype=np.float32)
    out[inside] = (1 - r[inside]**2)**3      # Poly6
    return out

# --- 3.  sweep over CLs -----------------------------------------------------
for tree, R in zip(trees, radii_per_cl):
    dists, idx = tree.query(coords, k=1)     # nearest sample along this CL
    w = dists / R[idx]                       # normalised distance
    contrib = kernel(w)
    field += contrib                         # accumulate

field = field.reshape(X.shape)               # back to (n,n,n)

print(field)

/var/folders/08/9rbxcpbs2rl1znd03wn5hz6m0000gn/T/ipykernel_53323/2728091596.py:27: RuntimeWarning:

divide by zero encountered in divide



KeyboardInterrupt: 

In [ ]:
# --- 4.  visualise ----------------------------------------------------------
import plotly.express as px

k = field.shape[2]//2          # mid-z slice
print(k)
px.imshow(field[:, :, 70], origin="lower",
          color_continuous_scale="Viridis",
          aspect="equal").show()

50
